# Calibrated mutation sample generation

Creates calibrated mutation samples under:

```text
Code_Submission/simulation_mutation/mutation_samples
```

The calibration chooses a common latent Gaussian-rank mutation intensity for each mutation family and desired final physical Pearson shift. It preserves empirical marginal distributions using the selected preservation mode and writes a sample-file index for the downstream simulation notebook.

CSV files cannot have sheets, so this notebook writes sample CSVs plus Excel **index workbooks** with one sheet per scenario level.

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata


# ============================================================
# SECTION 1. USER CONTROLS
# ============================================================

# Optional absolute Code_Submission override. Leave as None unless Jupyter resolves paths incorrectly.
# Example:
# code_root_override = "/absolute/path/to/Code for submission"
code_root_override = None

# Input folders/files, relative to Code_Submission unless absolute.
match_folder_name = str(Path("Input data and files") / "Match files of original data")
# Baseline realized samples supplied by the user:
#   Code_Submission/Input data and files/Simplified baseline realized samples
sample_root_dir = str(Path("Input data and files") / "Simplified baseline realized samples")

# Output folder required by the calibrated workflow:
# Code_Submission/simulation_mutation/mutation_samples
mutation_samples_dir_name = "mutation_samples"

# Mutation families:
#   "shape"            -> target pair rho(g,d), negative deterioration
#   "basis"            -> target pair rho(Ns,Nb_out), negative deterioration
#   "load_price"       -> target pair rho(d,Nb_out), positive intensification
#   "cannibalization"  -> target pair rho(g,Ns), negative intensification
enabled_mutation_families = ["shape", "basis", "load_price", "cannibalization"]

# Desired final physical Pearson shifts. The sign is taken from the family direction:
#   shape/basis/cannibalization use -0.10, -0.30, -0.50
#   load_price uses +0.10, +0.30, +0.50
desired_physical_shift_abs_levels = [0.10, 0.30, 0.50]

# Candidate latent Gaussian-rank deltas used to calibrate the desired final physical shift.
# Larger values may be needed for shape because quarter-hour marginal preservation is restrictive.
candidate_latent_abs_grid = [round(x, 2) for x in np.arange(0.00, 1.51, 0.05)]

# Calibration controls.
# Use a subset for calibration to avoid rerunning the sample generator many times.
# The final sample generation still uses all selected matches.
selected_match_ids = None                 # e.g. [1, 2, 3] or None for all discovered matches
calibration_selected_match_ids = None     # e.g. [1, 2, 3] or None to select automatically
calibration_max_matches = 30              # set None to use all selected matches
calibration_random_seed = 20260501
calibration_statistic = "median"          # "median" or "mean"
feasibility_tolerance = 0.05              # report target as infeasible if absolute error exceeds this

# Marginal preservation mode.
#   "quarter_hour": strictest; preserves empirical values inside each (quarter, hour) cell.
#   "quarter":      preserves empirical values inside each quarter, but allows hour-level reordering.
#   "annual":       preserves full-sample empirical values, but is the weakest temporal-control option.
preserve_marginals_mode = "quarter_hour"

# Final physical Pearson targeting controls.
# The latent Gaussian-rank calibration below is useful as a dependence-shape seed,
# but strict marginal preservation can attenuate the final physical Pearson shift.
# If True, each generated mutated sample is post-corrected so that, for each match,
# the final overall physical Pearson correlation of the target pair is as close as
# possible to: baseline physical Pearson correlation + target_physical_shift.
enforce_final_physical_shift_per_match = True
final_physical_shift_tolerance = 0.005

# Try the requested preservation mode first. If the target is mathematically
# unattainable under strict quarter-hour preservation, the code can relax to
# quarter-level or annual marginal preservation for the target-pair post-correction.
# Set this to [preserve_marginals_mode] to forbid relaxation.
physical_target_preserve_modes_to_try = ["quarter_hour", "quarter", "annual"]

# Search controls for the final physical Pearson post-correction. Larger values
# are slower but improve the chance of hitting tight per-match targets.
physical_target_rank_grid_size = 401
physical_target_random_seeds = 5
physical_target_swap_batches = 750
physical_target_swaps_per_batch = 150
physical_target_random_seed = 20260502

# Export controls.
write_mutated_sample_csvs = True
include_baseline_in_sample_index = True

# Baseline samples are normally not duplicated because they can be reconstructed from
# Simplified baseline realized samples. Set True only if you need explicit baseline CSV files.
write_baseline_sample_csvs = False

# Excel cannot store arbitrary huge simulation samples safely. The workbook export below
# writes scenario/sample-file index sheets, not the full sample banks.
write_excel_index_workbooks = True

# Overwrite existing generated sample CSVs so old large files are replaced by compact files.
# Set False only when you want to protect an existing compact sample set.
overwrite_existing_sample_csvs = True

# Compact sample export controls.
# The saved mutation-sample CSVs keep only columns needed by the simulation.
# Large path/debug fields such as sample_file are intentionally not written.
sample_columns_to_save = [
    "replication",
    "hour_index",
    "generation",
    "demand",
    "seller_lmp",
    "buyer_lmp_out",
    "buyer_lmp_in",
]

# Physical volume columns are constrained to be nonnegative in generated sample banks.
# Nodal price columns are allowed to retain signed values.
nonnegative_sample_columns = [
    "generation",
    "demand",
]

sample_integer_columns = ["replication", "hour_index"]
sample_columns_to_drop = ["sample_file"]
sample_decimal_places = 2

# Compact metadata/diagnostic table export controls.
metadata_decimal_places = 2
metadata_columns_to_drop = ["sample_file", "code_root"]


In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


In [ ]:
# ============================================================
# SECTION 2. PATH HELPERS AND INPUT LOADING
# ============================================================

NOTEBOOK_CWD = Path.cwd().resolve()

if code_root_override is not None:
    CODE_ROOT = Path(code_root_override).expanduser().resolve()
elif NOTEBOOK_CWD.name == "simulation_mutation":
    CODE_ROOT = NOTEBOOK_CWD.parent.resolve()
else:
    CODE_ROOT = NOTEBOOK_CWD.resolve()

SIM_MUTATION_ROOT = CODE_ROOT / "simulation_mutation"
MUTATION_SAMPLES_DIR = SIM_MUTATION_ROOT / mutation_samples_dir_name


def _clean_code(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return None
    return text


def resolve_existing_dir(*candidates) -> Path:
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        path = Path(candidate).expanduser()
        checked.append(path)
        if path.exists() and path.is_dir():
            return path.resolve()
    raise FileNotFoundError(
        "Could not resolve an existing directory. Tried:\n" +
        "\n".join(str(p) for p in checked)
    )


def candidate_roots(*anchors, max_parent_depth: int = 4):
    roots = []
    seen = set()
    for anchor in anchors:
        if anchor is None:
            continue
        p = Path(anchor).expanduser()
        if p.suffix:
            p = p.parent
        for root in [p, *list(p.parents)[:max_parent_depth]]:
            key = str(root)
            if key not in seen:
                seen.add(key)
                roots.append(root)
    return roots


def resolve_match_dir(match_folder_name: str) -> Path:
    roots = candidate_roots(CODE_ROOT, NOTEBOOK_CWD, Path("/mnt/data"))
    candidates = []
    for root in roots:
        candidates.extend([
            root / match_folder_name,
            root / "Input data and files" / match_folder_name,
            root / "Data and files" / match_folder_name,
        ])
    return resolve_existing_dir(*candidates)


def resolve_sample_root(sample_root_dir: str, match_dir: Path) -> Path:
    roots = candidate_roots(CODE_ROOT, NOTEBOOK_CWD, match_dir.parent, Path("/mnt/data"))
    candidates = []
    for root in roots:
        candidates.extend([
            root / sample_root_dir,
            root / "Input data and files" / sample_root_dir,
            root / "Data and files" / sample_root_dir,
        ])
    return resolve_existing_dir(*candidates)


def extract_match_id(path_like) -> int:
    path = Path(path_like)
    matched = re.search(r"(\d+)", path.stem)
    if matched is None:
        raise ValueError(f"Could not parse match_id from file name: {path.name}")
    return int(matched.group(1))


def discover_match_ids(match_dir: Path, selected_match_ids: Optional[Iterable[int]] = None) -> List[int]:
    match_ids = sorted({extract_match_id(path) for path in match_dir.glob("*.csv")})
    if selected_match_ids is not None:
        selected = {int(x) for x in selected_match_ids}
        match_ids = [mid for mid in match_ids if mid in selected]
    if not match_ids:
        raise ValueError("No matches were found under the historical match folder.")
    return match_ids


def discover_replication_dirs(sample_root: Path) -> List[Tuple[int, Path]]:
    rep_dirs = []
    for path in sample_root.iterdir():
        if path.is_dir():
            matched = re.search(r"rep_(\d+)", path.name, flags=re.IGNORECASE)
            if matched:
                rep_dirs.append((int(matched.group(1)), path))
    rep_dirs = sorted(rep_dirs, key=lambda x: x[0])
    if not rep_dirs:
        raise FileNotFoundError(
            f"No replication subfolders like rep_01, rep_02, ... were found under {sample_root}."
        )
    return rep_dirs


def enforce_nonnegative_sample_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Clip generated physical volume columns at zero; leave nodal price columns signed."""
    out = df.copy()
    for col in nonnegative_sample_columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").clip(lower=0.0)
    return out


def load_baseline_sample_bank(match_id: int, sample_root: Path) -> pd.DataFrame:
    frames = []
    rep_dirs = discover_replication_dirs(sample_root)

    for rep_no, rep_dir in rep_dirs:
        candidate = rep_dir / f"{int(match_id):03d}.csv"
        if not candidate.exists():
            alternatives = list(rep_dir.glob(f"*{int(match_id):03d}*.csv"))
            if alternatives:
                candidate = alternatives[0]
        if not candidate.exists():
            continue

        df = read_csv_optimized(candidate)
        required = ["timestamp", "hour", "quarter", "generation", "demand", "seller_lmp", "buyer_lmp_out", "buyer_lmp_in"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"{candidate} is missing required columns: {missing}")

        df = df.copy()
        df["replication"] = int(rep_no)
        df["hour_index"] = np.arange(1, len(df) + 1, dtype=int)
        df["sample_file"] = str(candidate)
        frames.append(df)

    if not frames:
        raise FileNotFoundError(
            f"No sample files were found for match_id={match_id} under {sample_root}."
        )

    bank = pd.concat(frames, ignore_index=True)
    bank = enforce_nonnegative_sample_columns(bank)
    bank["replication"] = pd.to_numeric(bank["replication"], errors="coerce").astype(int)
    bank["hour_index"] = pd.to_numeric(bank["hour_index"], errors="coerce").astype(int)
    return bank


def _rel_to_code_root(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(CODE_ROOT))
    except Exception:
        return str(Path(path).resolve())


In [ ]:
# ============================================================
# SECTION 3. MUTATION DEFINITIONS AND CORE METHODS
# ============================================================

VECTOR_COLUMNS = ["generation", "demand", "seller_lmp", "buyer_lmp_out"]
VECTOR_LABELS = {
    "generation": "g",
    "demand": "d",
    "seller_lmp": "Ns",
    "buyer_lmp_out": "Nb_out",
}

MUTATION_FAMILY_SPECS = {
    "shape": {
        "mutation_family_label": "Shape deterioration",
        "target_pair": ("generation", "demand"),
        "target_pair_label": "rho(g,d)",
        "direction": -1.0,
        "scenario_prefix": "ShapeDeterioration",
    },
    "basis": {
        "mutation_family_label": "Basis deterioration",
        "target_pair": ("seller_lmp", "buyer_lmp_out"),
        "target_pair_label": "rho(Ns,Nb_out)",
        "direction": -1.0,
        "scenario_prefix": "BasisDeterioration",
    },
    "load_price": {
        "mutation_family_label": "Buyer load-price intensification",
        "target_pair": ("demand", "buyer_lmp_out"),
        "target_pair_label": "rho(d,Nb_out)",
        "direction": +1.0,
        "scenario_prefix": "LoadPriceIntensification",
    },
    "cannibalization": {
        "mutation_family_label": "Seller-side cannibalization intensification",
        "target_pair": ("generation", "seller_lmp"),
        "target_pair_label": "rho(g,Ns)",
        "direction": -1.0,
        "scenario_prefix": "SellerCannibalization",
    },
}


def _delta_token(delta: float) -> str:
    sign = "p" if float(delta) >= 0 else "m"
    return f"{sign}{abs(float(delta)):.2f}".replace(".", "p")


def safe_corr(left, right) -> float:
    left = pd.to_numeric(pd.Series(left), errors="coerce")
    right = pd.to_numeric(pd.Series(right), errors="coerce")
    valid = left.notna() & right.notna()
    if valid.sum() <= 2:
        return np.nan
    x = left.loc[valid].to_numpy(dtype=float)
    y = right.loc[valid].to_numpy(dtype=float)
    if np.nanstd(x) <= 0 or np.nanstd(y) <= 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def _nearest_correlation_by_eigen_clip(matrix: np.ndarray,
                                       min_eigenvalue: float = 1e-8,
                                       max_iterations: int = 5) -> np.ndarray:
    C = np.asarray(matrix, dtype=float)
    C = 0.5 * (C + C.T)
    np.fill_diagonal(C, 1.0)

    for _ in range(int(max_iterations)):
        eigvals, eigvecs = np.linalg.eigh(C)
        eigvals = np.maximum(eigvals, float(min_eigenvalue))
        C = (eigvecs * eigvals) @ eigvecs.T
        C = 0.5 * (C + C.T)
        diag = np.sqrt(np.maximum(np.diag(C), float(min_eigenvalue)))
        C = C / np.outer(diag, diag)
        C = 0.5 * (C + C.T)
        np.fill_diagonal(C, 1.0)

    return C


def _matrix_sqrt_psd(matrix: np.ndarray, inverse: bool = False, eps: float = 1e-8) -> np.ndarray:
    C = _nearest_correlation_by_eigen_clip(matrix, min_eigenvalue=eps)
    eigvals, eigvecs = np.linalg.eigh(C)
    eigvals = np.maximum(eigvals, eps)
    weights = 1.0 / np.sqrt(eigvals) if inverse else np.sqrt(eigvals)
    return (eigvecs * weights) @ eigvecs.T


def _normal_scores(values: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=float)
    out = np.zeros_like(x, dtype=float)
    valid = np.isfinite(x)
    if valid.sum() <= 1 or np.nanstd(x[valid]) <= 0:
        return out
    ranks = rankdata(x[valid], method="average")
    u = ranks / (valid.sum() + 1.0)
    out[valid] = norm.ppf(u)
    return out


def _latent_matrix_from_values(df: pd.DataFrame, columns: List[str]) -> np.ndarray:
    return np.column_stack([
        _normal_scores(pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float))
        for col in columns
    ])


def _safe_corr_matrix_from_latent(Z: np.ndarray) -> np.ndarray:
    Z = np.asarray(Z, dtype=float)
    n_cols = Z.shape[1]
    if Z.shape[0] <= 2:
        return np.eye(n_cols)
    C = np.corrcoef(Z, rowvar=False)
    if not np.all(np.isfinite(C)):
        C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
        np.fill_diagonal(C, 1.0)
    C = 0.5 * (C + C.T)
    np.fill_diagonal(C, 1.0)
    return _nearest_correlation_by_eigen_clip(C)


def _assign_sorted_values_by_score(original_values: np.ndarray,
                                   target_scores: np.ndarray) -> np.ndarray:
    original_values = np.asarray(original_values, dtype=float)
    target_scores = np.asarray(target_scores, dtype=float)
    if len(original_values) <= 1:
        return original_values.copy()

    score_order = np.argsort(target_scores, kind="mergesort")
    sorted_values = np.sort(original_values, kind="mergesort")
    out = np.empty_like(original_values, dtype=float)
    out[score_order] = sorted_values
    return out


def _buyer_purchase_multiplier(sample_df: pd.DataFrame) -> np.ndarray:
    out = pd.to_numeric(sample_df["buyer_lmp_out"], errors="coerce").to_numpy(dtype=float)
    inn = pd.to_numeric(sample_df["buyer_lmp_in"], errors="coerce").to_numpy(dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        multiplier = inn / out
    finite = np.isfinite(multiplier) & (np.abs(out) > 1e-12)
    fill_value = float(np.nanmedian(multiplier[finite])) if finite.any() else 1.0
    multiplier = np.where(finite, multiplier, fill_value)
    multiplier = np.where(np.isfinite(multiplier), multiplier, fill_value)
    return multiplier


def build_scenario_spec(family: str,
                        target_physical_shift: float,
                        calibrated_latent_delta: float,
                        scenario_order: int,
                        calibration_error: float = np.nan,
                        calibration_feasible: bool = True) -> Dict[str, object]:
    spec = MUTATION_FAMILY_SPECS[family]
    scenario_name = (
        f"Mutation__{spec['scenario_prefix']}__"
        f"target_physical_shift_{_delta_token(target_physical_shift)}"
    )
    return {
        "scenario_name": scenario_name,
        "scenario_type": "mutation",
        "scenario_order": int(scenario_order),
        "mutation_family": family,
        "mutation_family_label": spec["mutation_family_label"],
        "target_pair": tuple(spec["target_pair"]),
        "target_pair_label": spec["target_pair_label"],
        "target_var_i": spec["target_pair"][0],
        "target_var_j": spec["target_pair"][1],
        "target_delta": float(calibrated_latent_delta),  # latent Gaussian-rank delta actually applied
        "calibrated_latent_delta": float(calibrated_latent_delta),
        "target_physical_shift": float(target_physical_shift),
        "desired_physical_shift_abs": abs(float(target_physical_shift)),
        "mutation_level_abs": abs(float(target_physical_shift)),
        "mutation_level_label": f"Target physical Delta rho {float(target_physical_shift):+.2f}",
        "mutation_method": "calibrated_gaussian_rank_recoloring_realized_bank",
        "calibration_error": float(calibration_error) if pd.notna(calibration_error) else np.nan,
        "calibration_feasible": bool(calibration_feasible),
        "preserve_marginals_mode": str(preserve_marginals_mode),
    }


def baseline_scenario_spec() -> Dict[str, object]:
    return {
        "scenario_name": "Baseline__No_Mutation__Verified",
        "scenario_type": "baseline",
        "scenario_order": 0,
        "mutation_family": "baseline",
        "mutation_family_label": "Baseline",
        "target_pair": None,
        "target_pair_label": "none",
        "target_var_i": "",
        "target_var_j": "",
        "target_delta": 0.0,
        "calibrated_latent_delta": 0.0,
        "target_physical_shift": 0.0,
        "desired_physical_shift_abs": 0.0,
        "mutation_level_abs": 0.0,
        "mutation_level_label": "Baseline",
        "mutation_method": "none",
        "calibration_error": 0.0,
        "calibration_feasible": True,
        "preserve_marginals_mode": str(preserve_marginals_mode),
    }


def _baseline_diagnostics(flat: pd.DataFrame, scenario_spec: Dict[str, object]) -> pd.DataFrame:
    diag_rows = []
    for quarter_value, q_df in flat.groupby("quarter", sort=True):
        Z_q = _latent_matrix_from_values(q_df, VECTOR_COLUMNS)
        C_q_latent = _safe_corr_matrix_from_latent(Z_q)
        for col_i, col_j, label in [
            ("generation", "demand", "rho(g,d)"),
            ("seller_lmp", "buyer_lmp_out", "rho(Ns,Nb_out)"),
            ("demand", "buyer_lmp_out", "rho(d,Nb_out)"),
            ("generation", "seller_lmp", "rho(g,Ns)"),
        ]:
            ii = VECTOR_COLUMNS.index(col_i)
            jj = VECTOR_COLUMNS.index(col_j)
            latent_corr = float(C_q_latent[ii, jj])
            diag_rows.append({
                **{k: scenario_spec.get(k) for k in [
                    "scenario_name", "scenario_type", "scenario_order", "mutation_family",
                    "mutation_family_label", "target_delta", "calibrated_latent_delta",
                    "target_physical_shift", "mutation_level_abs", "mutation_level_label",
                    "mutation_method", "preserve_marginals_mode"
                ]},
                "target_pair_label": label,
                "target_var_i": col_i,
                "target_var_j": col_j,
                "quarter": quarter_value,
                "n_rows_quarter": int(len(q_df)),
                "base_latent_corr": latent_corr,
                "pre_correction_target_corr": latent_corr,
                "post_correction_target_corr": latent_corr,
                "realized_physical_corr": safe_corr(q_df[col_i], q_df[col_j]),
                "realized_latent_corr": latent_corr,
                "psd_correction_frobenius": 0.0,
                "min_eigen_pre_correction": np.nan,
                "min_eigen_post_correction": np.nan,
                "base_physical_corr_global": safe_corr(flat[col_i], flat[col_j]),
                "final_physical_corr_global": safe_corr(flat[col_i], flat[col_j]),
                "final_physical_shift_global": 0.0,
            })
    return pd.DataFrame(diag_rows)


def mutate_sample_bank_by_gaussian_rank_recoloring(sample_df: pd.DataFrame,
                                                   scenario_spec: Dict[str, object],
                                                   preserve_mode: str = "quarter_hour") -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create one mutated scenario pool from the realized baseline pool.

    The mutation is imposed on latent Gaussian-rank dependence. The final values
    are empirical values reassigned according to target latent scores, so the chosen
    marginal preservation mode controls how much temporal structure is retained.
    """
    preserve_mode = str(preserve_mode).strip().lower()
    if preserve_mode not in {"quarter_hour", "quarter", "annual"}:
        raise ValueError("preserve_mode must be one of: 'quarter_hour', 'quarter', 'annual'.")

    flat = sample_df.sort_values(["replication", "hour_index"]).reset_index(drop=True).copy()
    flat["_buyer_purchase_multiplier"] = _buyer_purchase_multiplier(flat)

    if str(scenario_spec.get("scenario_type", "baseline")) == "baseline":
        diag = _baseline_diagnostics(flat.drop(columns=["_buyer_purchase_multiplier"]), scenario_spec)
        flat = flat.drop(columns=["_buyer_purchase_multiplier"])
        return flat, diag

    target_pair = tuple(scenario_spec["target_pair"])
    target_delta = float(scenario_spec["target_delta"])
    i = VECTOR_COLUMNS.index(target_pair[0])
    j = VECTOR_COLUMNS.index(target_pair[1])

    target_scores = {
        col: np.full(len(flat), np.nan, dtype=float)
        for col in VECTOR_COLUMNS
    }
    prelim_rows = []

    for quarter_value, q_positions_index in flat.groupby("quarter", sort=True).groups.items():
        q_pos = np.asarray(list(q_positions_index), dtype=int)
        q_df = flat.iloc[q_pos].copy()

        Z = _latent_matrix_from_values(q_df, VECTOR_COLUMNS)
        C_base = _safe_corr_matrix_from_latent(Z)

        C_pre = C_base.copy()
        C_pre[i, j] = np.clip(C_base[i, j] + target_delta, -1.0, 1.0)
        C_pre[j, i] = C_pre[i, j]
        C_pre = 0.5 * (C_pre + C_pre.T)
        np.fill_diagonal(C_pre, 1.0)

        eig_pre = np.linalg.eigvalsh(C_pre)
        C_post = _nearest_correlation_by_eigen_clip(C_pre)
        eig_post = np.linalg.eigvalsh(C_post)
        correction_frob = float(np.linalg.norm(C_post - C_pre, ord="fro"))

        Z_centered = Z - np.nanmean(Z, axis=0, keepdims=True)
        S_base = _safe_corr_matrix_from_latent(Z_centered)
        W = _matrix_sqrt_psd(S_base, inverse=True)
        T = _matrix_sqrt_psd(C_post, inverse=False)
        Z_target = Z_centered @ W @ T

        for col_idx, col in enumerate(VECTOR_COLUMNS):
            target_scores[col][q_pos] = Z_target[:, col_idx]

        prelim_rows.append({
            "quarter": quarter_value,
            "q_pos": q_pos,
            "base_latent_corr": float(C_base[i, j]),
            "pre_correction_target_corr": float(C_pre[i, j]),
            "post_correction_target_corr": float(C_post[i, j]),
            "psd_correction_frobenius": correction_frob,
            "min_eigen_pre_correction": float(np.min(eig_pre)),
            "min_eigen_post_correction": float(np.min(eig_post)),
            "n_rows_quarter": int(len(q_df)),
        })

    mutated_values = {
        col: pd.to_numeric(flat[col], errors="coerce").to_numpy(dtype=float).copy()
        for col in VECTOR_COLUMNS
    }

    def assign_group(local_pos: np.ndarray):
        for col in VECTOR_COLUMNS:
            mutated_values[col][local_pos] = _assign_sorted_values_by_score(
                original_values=flat[col].to_numpy(dtype=float)[local_pos],
                target_scores=target_scores[col][local_pos],
            )

    if preserve_mode == "quarter_hour":
        if "hour" not in flat.columns:
            raise KeyError("preserve_mode='quarter_hour' requires an 'hour' column.")
        for _, positions in flat.groupby(["quarter", "hour"], sort=False).groups.items():
            assign_group(np.asarray(list(positions), dtype=int))
    elif preserve_mode == "quarter":
        for _, positions in flat.groupby(["quarter"], sort=False).groups.items():
            assign_group(np.asarray(list(positions), dtype=int))
    else:  # annual/global empirical marginal preservation
        all_pos = np.arange(len(flat), dtype=int)
        assign_group(all_pos)

    for col in VECTOR_COLUMNS:
        flat[col] = mutated_values[col]

    flat["buyer_lmp_in"] = (
        flat["_buyer_purchase_multiplier"].to_numpy(dtype=float)
        * flat["buyer_lmp_out"].to_numpy(dtype=float)
    )
    flat = enforce_nonnegative_sample_columns(flat)

    diag_rows = []
    base_global = safe_corr(sample_df[target_pair[0]], sample_df[target_pair[1]])
    final_global = safe_corr(flat[target_pair[0]], flat[target_pair[1]])

    for row in prelim_rows:
        q_pos = row.pop("q_pos")
        q_base = sample_df.sort_values(["replication", "hour_index"]).reset_index(drop=True).iloc[q_pos].copy()
        q_mut = flat.iloc[q_pos].copy()
        Z_mut = _latent_matrix_from_values(q_mut, VECTOR_COLUMNS)
        C_mut_latent = _safe_corr_matrix_from_latent(Z_mut)

        diag_rows.append({
            **{k: scenario_spec.get(k) for k in [
                "scenario_name", "scenario_type", "scenario_order", "mutation_family",
                "mutation_family_label", "target_pair_label", "target_var_i", "target_var_j",
                "target_delta", "calibrated_latent_delta", "target_physical_shift",
                "desired_physical_shift_abs", "mutation_level_abs", "mutation_level_label",
                "mutation_method", "calibration_error", "calibration_feasible",
                "preserve_marginals_mode"
            ]},
            **row,
            "realized_physical_corr": safe_corr(q_mut[target_pair[0]], q_mut[target_pair[1]]),
            "realized_latent_corr": float(C_mut_latent[i, j]),
            "base_physical_corr_global": base_global,
            "final_physical_corr_global": final_global,
            "final_physical_shift_global": final_global - base_global if pd.notna(base_global) and pd.notna(final_global) else np.nan,
        })

    flat = flat.drop(columns=["_buyer_purchase_multiplier"])
    return flat, pd.DataFrame(diag_rows)


# ============================================================
# FINAL OVERALL PHYSICAL PEARSON TARGETING
# ============================================================

def _as_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def _blank_physical_target_report() -> Dict[str, object]:
    return {
        "physical_target_enforced": False,
        "physical_target_status": "not_applicable",
        "physical_target_base_corr": np.nan,
        "physical_target_requested_shift": np.nan,
        "physical_target_requested_corr": np.nan,
        "physical_target_requested_corr_in_unit_range": np.nan,
        "physical_target_final_corr": np.nan,
        "physical_target_final_shift": np.nan,
        "physical_target_error": np.nan,
        "physical_target_feasible": np.nan,
        "physical_target_preserve_mode_used": "",
        "physical_target_min_attainable_corr": np.nan,
        "physical_target_max_attainable_corr": np.nan,
        "physical_target_rank_grid_size": np.nan,
        "physical_target_random_seeds": np.nan,
        "physical_target_swap_batches_used": np.nan,
        "physical_target_method": "none",
    }


def _physical_target_modes_to_try(preferred_mode: str) -> List[str]:
    valid = {"quarter_hour", "quarter", "annual"}
    raw_modes = physical_target_preserve_modes_to_try
    if raw_modes is None:
        raw_modes = [preferred_mode]
    if isinstance(raw_modes, str):
        raw_modes = [raw_modes]

    out = []
    for mode in [preferred_mode, *list(raw_modes)]:
        mode_clean = str(mode).strip().lower()
        if mode_clean not in valid:
            raise ValueError("physical_target_preserve_modes_to_try must contain only: quarter_hour, quarter, annual.")
        if mode_clean not in out:
            out.append(mode_clean)
    return out


def _groups_for_preserve_mode(flat: pd.DataFrame, preserve_mode: str) -> List[np.ndarray]:
    preserve_mode = str(preserve_mode).strip().lower()
    if preserve_mode == "quarter_hour":
        required = ["quarter", "hour"]
        missing = [c for c in required if c not in flat.columns]
        if missing:
            raise KeyError(f"preserve_mode='quarter_hour' requires columns: {missing}")
        group_key = required
    elif preserve_mode == "quarter":
        if "quarter" not in flat.columns:
            raise KeyError("preserve_mode='quarter' requires a 'quarter' column.")
        group_key = "quarter"
    elif preserve_mode == "annual":
        return [np.arange(len(flat), dtype=int)]
    else:
        raise ValueError("preserve_mode must be one of: 'quarter_hour', 'quarter', 'annual'.")

    groups = []
    for _, positions in flat.groupby(group_key, sort=False).groups.items():
        groups.append(np.asarray(list(positions), dtype=int))
    return groups


def _assign_values_by_grouped_scores(source_values: np.ndarray,
                                     scores: np.ndarray,
                                     groups: List[np.ndarray]) -> np.ndarray:
    source_values = np.asarray(source_values, dtype=float)
    scores = np.asarray(scores, dtype=float)
    scores = np.where(np.isfinite(scores), scores, 0.0)
    out = source_values.copy()
    for pos in groups:
        if len(pos) <= 1:
            continue
        out[pos] = _assign_sorted_values_by_score(source_values[pos], scores[pos])
    return out


def _standardized_vector(values: np.ndarray) -> np.ndarray:
    z = np.asarray(values, dtype=float)
    finite = np.isfinite(z)
    out = np.zeros_like(z, dtype=float)
    if finite.sum() <= 1:
        return out
    out[finite] = z[finite] - np.mean(z[finite])
    std = np.std(out[finite])
    if std <= 1e-12 or not np.isfinite(std):
        return np.zeros_like(z, dtype=float)
    out[finite] = out[finite] / std
    return out


def _orthogonal_noise(z_x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    z_x = _standardized_vector(z_x)
    noise = rng.normal(size=len(z_x))
    noise = _standardized_vector(noise)
    denom = float(np.dot(z_x, z_x))
    if denom > 1e-12:
        noise = noise - float(np.dot(noise, z_x)) / denom * z_x
    noise = _standardized_vector(noise)
    return noise


def _corr_error_for_target(base_corr: float, final_corr: float, requested_shift: float) -> float:
    if pd.isna(base_corr) or pd.isna(final_corr):
        return np.nan
    return abs((float(final_corr) - float(base_corr)) - float(requested_shift))


def _refine_assignment_by_random_swaps(x_values: np.ndarray,
                                       y_values: np.ndarray,
                                       groups: List[np.ndarray],
                                       target_corr: float,
                                       tolerance: float,
                                       rng: np.random.Generator) -> Tuple[np.ndarray, float, int]:
    """Improve a grouped y assignment by swaps that move the global Pearson correlation toward target_corr."""
    y = np.asarray(y_values, dtype=float).copy()
    x = np.asarray(x_values, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() <= 2:
        return y, safe_corr(x, y), 0

    x_center = np.zeros_like(x, dtype=float)
    x_center[valid] = x[valid] - np.mean(x[valid])
    y_mean = float(np.mean(y[valid]))
    y_center = np.zeros_like(y, dtype=float)
    y_center[valid] = y[valid] - y_mean

    denom = math.sqrt(float(np.dot(x_center[valid], x_center[valid])) * float(np.dot(y_center[valid], y_center[valid])))
    if denom <= 1e-12 or not np.isfinite(denom):
        return y, safe_corr(x, y), 0

    target_num = float(target_corr) * denom
    current_num = float(np.dot(x_center[valid], y_center[valid]))
    current_error = abs(current_num - target_num)
    target_num_tolerance = float(tolerance) * denom

    swap_groups = [pos[valid[pos]] for pos in groups if np.sum(valid[pos]) >= 2]
    if not swap_groups:
        return y, current_num / denom, 0

    weights = np.asarray([len(pos) for pos in swap_groups], dtype=float)
    weights = weights / weights.sum()

    batches_used = 0
    max_batches = int(physical_target_swap_batches)
    swaps_per_batch = int(physical_target_swaps_per_batch)

    for batch in range(max_batches):
        batches_used = batch + 1
        if current_error <= target_num_tolerance:
            break

        best_error = current_error
        best_pair = None
        best_delta = 0.0

        for _ in range(swaps_per_batch):
            group_idx = int(rng.choice(len(swap_groups), p=weights))
            pos = swap_groups[group_idx]
            local_a = int(rng.integers(0, len(pos)))
            local_b = int(rng.integers(0, len(pos) - 1))
            if local_b >= local_a:
                local_b += 1
            a = int(pos[local_a])
            b = int(pos[local_b])
            if y[a] == y[b]:
                continue

            delta_num = (x_center[a] - x_center[b]) * (y[b] - y[a])
            new_error = abs((current_num + delta_num) - target_num)
            if new_error + 1e-15 < best_error:
                best_error = new_error
                best_pair = (a, b)
                best_delta = float(delta_num)

        if best_pair is None:
            break

        a, b = best_pair
        y[a], y[b] = y[b], y[a]
        current_num += best_delta
        current_error = best_error

    return y, float(current_num / denom), int(batches_used)


def _try_final_physical_corr_target(mut_flat: pd.DataFrame,
                                    base_flat: pd.DataFrame,
                                    scenario_spec: Dict[str, object],
                                    preserve_mode: str,
                                    match_id: int) -> Tuple[pd.DataFrame, Dict[str, object]]:
    out = mut_flat.copy()
    target_pair = tuple(scenario_spec["target_pair"])
    x_col, y_col = target_pair[0], target_pair[1]
    requested_shift = float(scenario_spec.get("target_physical_shift", 0.0))

    base_corr = safe_corr(base_flat[x_col], base_flat[y_col])
    target_corr = float(base_corr + requested_shift) if pd.notna(base_corr) else np.nan
    requested_in_unit_range = bool(pd.notna(target_corr) and (-1.0 <= target_corr <= 1.0))

    report = _blank_physical_target_report()
    report.update({
        "physical_target_enforced": True,
        "physical_target_status": "attempted",
        "physical_target_base_corr": float(base_corr) if pd.notna(base_corr) else np.nan,
        "physical_target_requested_shift": requested_shift,
        "physical_target_requested_corr": float(target_corr) if pd.notna(target_corr) else np.nan,
        "physical_target_requested_corr_in_unit_range": requested_in_unit_range,
        "physical_target_preserve_mode_used": str(preserve_mode),
        "physical_target_rank_grid_size": int(physical_target_rank_grid_size),
        "physical_target_random_seeds": int(physical_target_random_seeds),
        "physical_target_method": "grouped_rank_reassignment_plus_swap_refinement",
    })

    if pd.isna(base_corr) or pd.isna(target_corr):
        final_corr = safe_corr(out[x_col], out[y_col])
        report.update({
            "physical_target_status": "failed_nan_base_or_target_corr",
            "physical_target_final_corr": final_corr,
            "physical_target_final_shift": final_corr - base_corr if pd.notna(base_corr) and pd.notna(final_corr) else np.nan,
            "physical_target_error": _corr_error_for_target(base_corr, final_corr, requested_shift),
            "physical_target_feasible": False,
        })
        return out, report

    groups = _groups_for_preserve_mode(out, preserve_mode)
    x_values = pd.to_numeric(out[x_col], errors="coerce").to_numpy(dtype=float)
    y_source = pd.to_numeric(out[y_col], errors="coerce").to_numpy(dtype=float)

    y_min = _assign_values_by_grouped_scores(y_source, -x_values, groups)
    y_max = _assign_values_by_grouped_scores(y_source, x_values, groups)
    min_corr = safe_corr(x_values, y_min)
    max_corr = safe_corr(x_values, y_max)
    if pd.notna(min_corr) and pd.notna(max_corr) and min_corr > max_corr:
        min_corr, max_corr = max_corr, min_corr
        y_min, y_max = y_max, y_min

    report.update({
        "physical_target_min_attainable_corr": float(min_corr) if pd.notna(min_corr) else np.nan,
        "physical_target_max_attainable_corr": float(max_corr) if pd.notna(max_corr) else np.nan,
    })

    if pd.isna(min_corr) or pd.isna(max_corr):
        final_corr = safe_corr(x_values, y_source)
        report.update({
            "physical_target_status": "failed_nan_attainable_range",
            "physical_target_final_corr": final_corr,
            "physical_target_final_shift": final_corr - base_corr if pd.notna(final_corr) else np.nan,
            "physical_target_error": _corr_error_for_target(base_corr, final_corr, requested_shift),
            "physical_target_feasible": False,
        })
        return out, report

    target_in_attainable_range = (float(min_corr) - float(final_physical_shift_tolerance) <= target_corr <= float(max_corr) + float(final_physical_shift_tolerance))
    if not requested_in_unit_range or not target_in_attainable_range:
        if abs(float(target_corr) - float(min_corr)) <= abs(float(target_corr) - float(max_corr)):
            best_y = y_min
            final_corr = min_corr
        else:
            best_y = y_max
            final_corr = max_corr
        out[y_col] = best_y
        if y_col == "buyer_lmp_out" and "buyer_lmp_in" in out.columns:
            out["buyer_lmp_in"] = _buyer_purchase_multiplier(base_flat) * out["buyer_lmp_out"].to_numpy(dtype=float)
        out = enforce_nonnegative_sample_columns(out)
        report.update({
            "physical_target_status": "outside_attainable_range",
            "physical_target_final_corr": float(final_corr),
            "physical_target_final_shift": float(final_corr - base_corr),
            "physical_target_error": _corr_error_for_target(base_corr, final_corr, requested_shift),
            "physical_target_feasible": False,
        })
        return out, report

    # Candidate rank-score search. Endpoints are included, and several orthogonal-noise
    # bridges make the attainable correlation grid much denser than the latent-delta grid.
    z_x = _normal_scores(x_values)
    z_x = _standardized_vector(z_x)
    alpha_grid = np.linspace(-1.0, 1.0, int(physical_target_rank_grid_size))
    seed_base = int(physical_target_random_seed) + int(match_id) * 1009 + int(scenario_spec.get("scenario_order", 0)) * 917

    best_y = y_source.copy()
    best_corr = safe_corr(x_values, best_y)
    best_error = abs(float(best_corr) - float(target_corr)) if pd.notna(best_corr) else np.inf

    for seed_idx in range(max(1, int(physical_target_random_seeds))):
        rng = np.random.default_rng(seed_base + seed_idx)
        noise = _orthogonal_noise(z_x, rng)
        for alpha in alpha_grid:
            beta = math.sqrt(max(0.0, 1.0 - float(alpha) ** 2))
            scores = float(alpha) * z_x + beta * noise
            candidate_y = _assign_values_by_grouped_scores(y_source, scores, groups)
            candidate_corr = safe_corr(x_values, candidate_y)
            if pd.isna(candidate_corr):
                continue
            candidate_error = abs(float(candidate_corr) - float(target_corr))
            if candidate_error < best_error:
                best_y = candidate_y
                best_corr = float(candidate_corr)
                best_error = candidate_error
                if best_error <= float(final_physical_shift_tolerance):
                    break
        if best_error <= float(final_physical_shift_tolerance):
            break

    rng_refine = np.random.default_rng(seed_base + 1000003)
    refined_y, refined_corr, batches_used = _refine_assignment_by_random_swaps(
        x_values=x_values,
        y_values=best_y,
        groups=groups,
        target_corr=target_corr,
        tolerance=float(final_physical_shift_tolerance),
        rng=rng_refine,
    )
    refined_error = abs(float(refined_corr) - float(target_corr)) if pd.notna(refined_corr) else np.inf
    if refined_error <= best_error or pd.isna(best_corr):
        best_y = refined_y
        best_corr = refined_corr
        best_error = refined_error

    out[y_col] = best_y
    if y_col == "buyer_lmp_out" and "buyer_lmp_in" in out.columns:
        out["buyer_lmp_in"] = _buyer_purchase_multiplier(base_flat) * out["buyer_lmp_out"].to_numpy(dtype=float)
    out = enforce_nonnegative_sample_columns(out)

    final_corr = safe_corr(out[x_col], out[y_col])
    final_shift = final_corr - base_corr if pd.notna(final_corr) else np.nan
    final_error = _corr_error_for_target(base_corr, final_corr, requested_shift)
    feasible = bool(requested_in_unit_range and target_in_attainable_range and pd.notna(final_error) and final_error <= float(final_physical_shift_tolerance))
    report.update({
        "physical_target_status": "success" if feasible else "attempted_not_within_tolerance",
        "physical_target_final_corr": float(final_corr) if pd.notna(final_corr) else np.nan,
        "physical_target_final_shift": float(final_shift) if pd.notna(final_shift) else np.nan,
        "physical_target_error": float(final_error) if pd.notna(final_error) else np.nan,
        "physical_target_feasible": feasible,
        "physical_target_swap_batches_used": int(batches_used),
    })
    return out, report


def enforce_final_physical_shift_for_match(base_df: pd.DataFrame,
                                           mut_df: pd.DataFrame,
                                           scenario_spec: Dict[str, object],
                                           preferred_preserve_mode: str,
                                           match_id: int) -> Tuple[pd.DataFrame, Dict[str, object]]:
    if not _as_bool(enforce_final_physical_shift_per_match):
        report = _blank_physical_target_report()
        target_pair = tuple(scenario_spec["target_pair"])
        x_col, y_col = target_pair[0], target_pair[1]
        base_flat = base_df.sort_values(["replication", "hour_index"]).reset_index(drop=True).copy()
        mut_flat = mut_df.sort_values(["replication", "hour_index"]).reset_index(drop=True).copy()
        base_corr = safe_corr(base_flat[x_col], base_flat[y_col])
        final_corr = safe_corr(mut_flat[x_col], mut_flat[y_col])
        requested_shift = float(scenario_spec.get("target_physical_shift", 0.0))
        report.update({
            "physical_target_status": "not_enforced",
            "physical_target_base_corr": base_corr,
            "physical_target_requested_shift": requested_shift,
            "physical_target_requested_corr": base_corr + requested_shift if pd.notna(base_corr) else np.nan,
            "physical_target_final_corr": final_corr,
            "physical_target_final_shift": final_corr - base_corr if pd.notna(base_corr) and pd.notna(final_corr) else np.nan,
            "physical_target_error": _corr_error_for_target(base_corr, final_corr, requested_shift),
            "physical_target_feasible": False,
        })
        return enforce_nonnegative_sample_columns(mut_df), report

    base_flat = enforce_nonnegative_sample_columns(
        base_df.sort_values(["replication", "hour_index"]).reset_index(drop=True)
    )
    mut_flat = enforce_nonnegative_sample_columns(
        mut_df.sort_values(["replication", "hour_index"]).reset_index(drop=True)
    )

    best_df = mut_flat.copy()
    best_report = None
    for mode in _physical_target_modes_to_try(preferred_preserve_mode):
        candidate_df, candidate_report = _try_final_physical_corr_target(
            mut_flat=mut_flat,
            base_flat=base_flat,
            scenario_spec=scenario_spec,
            preserve_mode=mode,
            match_id=int(match_id),
        )
        if best_report is None or (
            pd.notna(candidate_report.get("physical_target_error")) and
            (pd.isna(best_report.get("physical_target_error")) or
             float(candidate_report["physical_target_error"]) < float(best_report["physical_target_error"]))
        ):
            best_df = candidate_df
            best_report = candidate_report

        if bool(candidate_report.get("physical_target_feasible", False)):
            best_df = candidate_df
            best_report = candidate_report
            break

    if best_report is None:
        best_report = _blank_physical_target_report()
        best_report.update({
            "physical_target_enforced": True,
            "physical_target_status": "failed_no_modes_attempted",
            "physical_target_feasible": False,
        })
    return enforce_nonnegative_sample_columns(best_df), best_report


def refresh_mutation_diagnostics_after_physical_targeting(base_df: pd.DataFrame,
                                                          mut_df: pd.DataFrame,
                                                          scenario_spec: Dict[str, object],
                                                          diag: pd.DataFrame,
                                                          report: Dict[str, object]) -> pd.DataFrame:
    out_diag = diag.copy()
    for key, value in report.items():
        out_diag[key] = value

    if out_diag.empty:
        return out_diag

    target_pair = tuple(scenario_spec["target_pair"])
    x_col, y_col = target_pair[0], target_pair[1]
    i = VECTOR_COLUMNS.index(x_col)
    j = VECTOR_COLUMNS.index(y_col)

    base_flat = base_df.sort_values(["replication", "hour_index"]).reset_index(drop=True).copy()
    mut_flat = mut_df.sort_values(["replication", "hour_index"]).reset_index(drop=True).copy()
    base_global = safe_corr(base_flat[x_col], base_flat[y_col])
    final_global = safe_corr(mut_flat[x_col], mut_flat[y_col])

    out_diag["base_physical_corr_global"] = base_global
    out_diag["final_physical_corr_global"] = final_global
    out_diag["final_physical_shift_global"] = final_global - base_global if pd.notna(base_global) and pd.notna(final_global) else np.nan

    if "quarter" in out_diag.columns and "quarter" in mut_flat.columns:
        for quarter_value in out_diag["quarter"].dropna().unique():
            q_mask = mut_flat["quarter"].eq(quarter_value)
            if not q_mask.any():
                continue
            q_mut = mut_flat.loc[q_mask].copy()
            Z_mut = _latent_matrix_from_values(q_mut, VECTOR_COLUMNS)
            C_mut_latent = _safe_corr_matrix_from_latent(Z_mut)
            diag_mask = out_diag["quarter"].eq(quarter_value)
            out_diag.loc[diag_mask, "realized_physical_corr"] = safe_corr(q_mut[x_col], q_mut[y_col])
            out_diag.loc[diag_mask, "realized_latent_corr"] = float(C_mut_latent[i, j])

    return out_diag


In [ ]:
# ============================================================
# SECTION 4. CALIBRATION
# ============================================================

def _select_calibration_match_ids(match_ids: List[int]) -> List[int]:
    if calibration_selected_match_ids is not None:
        requested = {int(x) for x in calibration_selected_match_ids}
        out = [mid for mid in match_ids if mid in requested]
        if not out:
            raise ValueError("calibration_selected_match_ids did not match any available match IDs.")
        return out

    if calibration_max_matches is None or int(calibration_max_matches) >= len(match_ids):
        return list(match_ids)

    rng = np.random.default_rng(int(calibration_random_seed))
    selected = sorted(rng.choice(match_ids, size=int(calibration_max_matches), replace=False).tolist())
    return [int(x) for x in selected]


def _candidate_latent_deltas_for_family(family: str) -> List[float]:
    direction = float(MUTATION_FAMILY_SPECS[family]["direction"])
    return [float(direction * abs_delta) for abs_delta in candidate_latent_abs_grid]


def _family_desired_physical_shifts(family: str) -> List[float]:
    direction = float(MUTATION_FAMILY_SPECS[family]["direction"])
    return [float(direction * abs_delta) for abs_delta in desired_physical_shift_abs_levels]


def _calibration_stat(values: List[float]) -> float:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return np.nan
    if str(calibration_statistic).lower() == "mean":
        return float(np.mean(arr))
    return float(np.median(arr))


def calibrate_latent_deltas(match_ids_for_calibration: List[int],
                            sample_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    response_rows = []

    for family in enabled_mutation_families:
        spec_base = MUTATION_FAMILY_SPECS[family]
        target_pair = tuple(spec_base["target_pair"])

        print(f"Calibrating family: {family}")
        for candidate_delta in _candidate_latent_deltas_for_family(family):
            shifts = []
            final_corrs = []
            base_corrs = []

            scenario_spec = build_scenario_spec(
                family=family,
                target_physical_shift=0.0,
                calibrated_latent_delta=float(candidate_delta),
                scenario_order=999,
                calibration_error=np.nan,
                calibration_feasible=True,
            )
            scenario_spec["scenario_name"] = f"Calibration__{family}__latent_delta_{_delta_token(candidate_delta)}"

            for match_id in match_ids_for_calibration:
                base_df = load_baseline_sample_bank(int(match_id), sample_root)
                base_corr = safe_corr(base_df[target_pair[0]], base_df[target_pair[1]])
                mut_df, _ = mutate_sample_bank_by_gaussian_rank_recoloring(
                    sample_df=base_df,
                    scenario_spec=scenario_spec,
                    preserve_mode=preserve_marginals_mode,
                )
                final_corr = safe_corr(mut_df[target_pair[0]], mut_df[target_pair[1]])
                shift = final_corr - base_corr if pd.notna(base_corr) and pd.notna(final_corr) else np.nan
                shifts.append(shift)
                base_corrs.append(base_corr)
                final_corrs.append(final_corr)

            response_rows.append({
                "mutation_family": family,
                "mutation_family_label": spec_base["mutation_family_label"],
                "target_pair_label": spec_base["target_pair_label"],
                "target_var_i": target_pair[0],
                "target_var_j": target_pair[1],
                "candidate_latent_delta": float(candidate_delta),
                "candidate_latent_abs": abs(float(candidate_delta)),
                "n_calibration_matches": int(len(match_ids_for_calibration)),
                "preserve_marginals_mode": preserve_marginals_mode,
                "calibration_statistic": calibration_statistic,
                "mean_base_physical_corr": float(np.nanmean(base_corrs)),
                "mean_final_physical_corr": float(np.nanmean(final_corrs)),
                "mean_final_physical_shift": float(np.nanmean(shifts)),
                "median_final_physical_shift": float(np.nanmedian(shifts)),
                "p10_final_physical_shift": float(np.nanpercentile(shifts, 10)),
                "p90_final_physical_shift": float(np.nanpercentile(shifts, 90)),
                "min_final_physical_shift": float(np.nanmin(shifts)),
                "max_final_physical_shift": float(np.nanmax(shifts)),
            })

    response_df = pd.DataFrame(response_rows)

    selected_rows = []
    stat_col = "mean_final_physical_shift" if str(calibration_statistic).lower() == "mean" else "median_final_physical_shift"

    for family in enabled_mutation_families:
        fam_resp = response_df.loc[response_df["mutation_family"].astype(str).eq(family)].copy()
        for desired_shift in _family_desired_physical_shifts(family):
            fam_resp["abs_error_to_desired"] = (fam_resp[stat_col] - float(desired_shift)).abs()
            best = fam_resp.sort_values(["abs_error_to_desired", "candidate_latent_abs"]).iloc[0].to_dict()
            err = float(best["abs_error_to_desired"])
            feasible = bool(err <= float(feasibility_tolerance))
            selected_rows.append({
                "mutation_family": family,
                "mutation_family_label": MUTATION_FAMILY_SPECS[family]["mutation_family_label"],
                "target_pair_label": MUTATION_FAMILY_SPECS[family]["target_pair_label"],
                "target_var_i": MUTATION_FAMILY_SPECS[family]["target_pair"][0],
                "target_var_j": MUTATION_FAMILY_SPECS[family]["target_pair"][1],
                "desired_physical_shift": float(desired_shift),
                "desired_physical_shift_abs": abs(float(desired_shift)),
                "selected_latent_delta": float(best["candidate_latent_delta"]),
                "selected_latent_abs": abs(float(best["candidate_latent_delta"])),
                "achieved_calibration_stat_shift": float(best[stat_col]),
                "calibration_error": err,
                "calibration_feasible": feasible,
                "feasibility_tolerance": float(feasibility_tolerance),
                "calibration_statistic": calibration_statistic,
                "preserve_marginals_mode": preserve_marginals_mode,
                "n_calibration_matches": int(best["n_calibration_matches"]),
            })

    selected_df = pd.DataFrame(selected_rows)
    return response_df, selected_df


In [ ]:
# ============================================================
# SECTION 5. FINAL SAMPLE GENERATION AND EXPORT
# ============================================================

def build_calibrated_scenario_table(selected_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if include_baseline_in_sample_index:
        rows.append(baseline_scenario_spec())

    scenario_order = 1
    for _, r in selected_df.sort_values(["mutation_family", "desired_physical_shift_abs"]).iterrows():
        family = str(r["mutation_family"])
        desired_shift = float(r["desired_physical_shift"])
        selected_latent = float(r["selected_latent_delta"])
        rows.append(build_scenario_spec(
            family=family,
            target_physical_shift=desired_shift,
            calibrated_latent_delta=selected_latent,
            scenario_order=scenario_order,
            calibration_error=float(r["calibration_error"]),
            calibration_feasible=bool(r["calibration_feasible"]),
        ))
        scenario_order += 1

    return pd.DataFrame(rows)


def _sample_output_subdir(scenario_spec: Dict[str, object]) -> Path:
    if str(scenario_spec.get("scenario_type")) == "baseline":
        return MUTATION_SAMPLES_DIR / "baseline"
    family = str(scenario_spec["mutation_family"])
    token = _delta_token(float(scenario_spec["target_physical_shift"]))
    return MUTATION_SAMPLES_DIR / family / f"target_physical_shift_{token}"


def _sample_output_file(match_id: int, scenario_spec: Dict[str, object]) -> Path:
    subdir = _sample_output_subdir(scenario_spec)
    scenario_name = str(scenario_spec["scenario_name"])
    return subdir / f"Match_{int(match_id):04d}__{scenario_name}.csv"


def _compact_table_for_export(
    df: pd.DataFrame,
    *,
    drop_columns: Optional[List[str]] = None,
    decimal_places: Optional[int] = None,
) -> pd.DataFrame:
    """Return a compact copy for CSV/Excel export without altering in-memory values."""
    out = df.copy()
    if drop_columns:
        out = out.drop(columns=[c for c in drop_columns if c in out.columns], errors="ignore")

    if decimal_places is not None:
        int_cols = [c for c in sample_integer_columns if c in out.columns]
        for col in int_cols:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

        numeric_cols = [
            c for c in out.select_dtypes(include=[np.number]).columns
            if c not in int_cols
        ]
        if numeric_cols:
            out[numeric_cols] = out[numeric_cols].round(int(decimal_places))

    return out


def _write_csv_compact(
    df: pd.DataFrame,
    path: Path,
    *,
    drop_columns: Optional[List[str]] = None,
    decimal_places: Optional[int] = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    out = _compact_table_for_export(
        df,
        drop_columns=drop_columns,
        decimal_places=decimal_places,
    )
    float_format = None if decimal_places is None else f"%.{int(decimal_places)}f"
    out.to_csv(path, index=False, float_format=float_format)


def _write_sample_csv(sample_df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite_existing_sample_csvs:
        return

    out_df = enforce_nonnegative_sample_columns(sample_df).drop(
        columns=[c for c in sample_columns_to_drop if c in sample_df.columns],
        errors="ignore",
    ).copy()

    if sample_columns_to_save is not None:
        keep = [c for c in sample_columns_to_save if c in out_df.columns]
        missing = [c for c in sample_columns_to_save if c not in out_df.columns]
        if missing:
            raise KeyError(f"Sample export is missing required compact columns: {missing}")
        out_df = out_df[keep].copy()

    _write_csv_compact(
        out_df,
        path,
        drop_columns=sample_columns_to_drop,
        decimal_places=sample_decimal_places,
    )


def write_index_workbooks(sample_index_df: pd.DataFrame) -> None:
    if not write_excel_index_workbooks:
        return

    workbook_dir = MUTATION_SAMPLES_DIR / "index_workbooks"
    workbook_dir.mkdir(parents=True, exist_ok=True)

    # One workbook per mutation family. Each sheet gives the file index for one level.
    for family, fam_df in sample_index_df.groupby("mutation_family", dropna=False):
        family_name = str(family)
        if family_name == "baseline":
            continue
        wb_path = workbook_dir / f"Mutation_Sample_Index__{family_name}.xlsx"

        with pd.ExcelWriter(wb_path, engine="openpyxl") as writer:
            baseline_rows = sample_index_df.loc[sample_index_df["scenario_type"].astype(str).eq("baseline")].copy()
            if not baseline_rows.empty:
                _compact_table_for_export(baseline_rows, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places).to_excel(writer, sheet_name="baseline_index", index=False)

            fam_df = fam_df.sort_values(["desired_physical_shift_abs", "match_id"])
            for desired_abs, level_df in fam_df.groupby("desired_physical_shift_abs", dropna=False):
                shift = pd.to_numeric(level_df["target_physical_shift"], errors="coerce").dropna()
                sheet_token = _delta_token(float(shift.iloc[0])) if not shift.empty else str(desired_abs)
                sheet_name = f"shift_{sheet_token}"[:31]
                _compact_table_for_export(level_df, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places).to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"Saved index workbook: {wb_path}")


def generate_final_samples(match_ids: List[int],
                           sample_root: Path,
                           scenario_table_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    index_rows = []
    diag_rows = []
    status_rows = []

    scenario_records = scenario_table_df.to_dict("records")

    for counter, match_id in enumerate(match_ids, start=1):
        print(f"[{counter}/{len(match_ids)}] match_id={int(match_id):04d}")
        try:
            base_df = load_baseline_sample_bank(int(match_id), sample_root)
        except Exception as exc:
            status_rows.append({
                "match_id": int(match_id),
                "scenario_name": "",
                "status": "failed_loading_baseline_sample",
                "error_message": str(exc),
            })
            print(f"    failed loading baseline sample: {exc}")
            continue

        for scenario_spec in scenario_records:
            scenario_name = str(scenario_spec["scenario_name"])
            scenario_type = str(scenario_spec["scenario_type"])
            physical_target_report = _blank_physical_target_report()

            try:
                if scenario_type == "baseline":
                    if write_baseline_sample_csvs:
                        sample_path = _sample_output_file(int(match_id), scenario_spec)
                        if write_mutated_sample_csvs:
                            _write_sample_csv(base_df, sample_path)
                        sample_file = _rel_to_code_root(sample_path)
                        sample_source = "saved_baseline_csv"
                    else:
                        sample_file = ""
                        sample_source = "baseline_sample_root"

                    diag = _baseline_diagnostics(base_df, scenario_spec)
                    diag["match_id"] = int(match_id)
                    for key, value in physical_target_report.items():
                        diag[key] = value
                    diag_rows.extend(diag.to_dict("records"))

                else:
                    mut_df, diag = mutate_sample_bank_by_gaussian_rank_recoloring(
                        sample_df=base_df,
                        scenario_spec=scenario_spec,
                        preserve_mode=preserve_marginals_mode,
                    )

                    if _as_bool(enforce_final_physical_shift_per_match):
                        mut_df, physical_target_report = enforce_final_physical_shift_for_match(
                            base_df=base_df,
                            mut_df=mut_df,
                            scenario_spec=scenario_spec,
                            preferred_preserve_mode=preserve_marginals_mode,
                            match_id=int(match_id),
                        )
                        diag = refresh_mutation_diagnostics_after_physical_targeting(
                            base_df=base_df,
                            mut_df=mut_df,
                            scenario_spec=scenario_spec,
                            diag=diag,
                            report=physical_target_report,
                        )
                    else:
                        mut_df, physical_target_report = enforce_final_physical_shift_for_match(
                            base_df=base_df,
                            mut_df=mut_df,
                            scenario_spec=scenario_spec,
                            preferred_preserve_mode=preserve_marginals_mode,
                            match_id=int(match_id),
                        )
                        diag = refresh_mutation_diagnostics_after_physical_targeting(
                            base_df=base_df,
                            mut_df=mut_df,
                            scenario_spec=scenario_spec,
                            diag=diag,
                            report=physical_target_report,
                        )

                    sample_path = _sample_output_file(int(match_id), scenario_spec)
                    if write_mutated_sample_csvs:
                        _write_sample_csv(mut_df, sample_path)
                    sample_file = _rel_to_code_root(sample_path)
                    sample_source = "mutation_samples_csv"

                    diag["match_id"] = int(match_id)
                    diag_rows.extend(diag.to_dict("records"))

                row = dict(scenario_spec)
                row.pop("target_pair", None)
                row["match_id"] = int(match_id)
                row["sample_source"] = sample_source
                row["sample_path_rel"] = sample_file
                row.update(physical_target_report)
                index_rows.append(row)

                status_row = {
                    "match_id": int(match_id),
                    "scenario_name": scenario_name,
                    "scenario_type": scenario_type,
                    "mutation_family": scenario_spec.get("mutation_family"),
                    "target_physical_shift": scenario_spec.get("target_physical_shift"),
                    "calibrated_latent_delta": scenario_spec.get("calibrated_latent_delta"),
                    "status": "success",
                    "error_message": "",
                }
                status_row.update(physical_target_report)
                status_rows.append(status_row)

                target_msg = ""
                if scenario_type != "baseline" and _as_bool(enforce_final_physical_shift_per_match):
                    target_msg = (
                        f", final_shift={physical_target_report.get('physical_target_final_shift', np.nan):+.4f}, "
                        f"target_error={physical_target_report.get('physical_target_error', np.nan):.4f}, "
                        f"target_mode={physical_target_report.get('physical_target_preserve_mode_used', '')}, "
                        f"target_status={physical_target_report.get('physical_target_status', '')}"
                    )

                print(
                    f"    {scenario_name}: "
                    f"source={sample_source}, file={sample_file if sample_file else '[baseline root]'}"
                    f"{target_msg}"
                )

            except Exception as exc:
                status_row = {
                    "match_id": int(match_id),
                    "scenario_name": scenario_name,
                    "scenario_type": scenario_type,
                    "mutation_family": scenario_spec.get("mutation_family"),
                    "target_physical_shift": scenario_spec.get("target_physical_shift"),
                    "calibrated_latent_delta": scenario_spec.get("calibrated_latent_delta"),
                    "status": "failed_generating_sample",
                    "error_message": str(exc),
                }
                status_row.update(physical_target_report)
                status_rows.append(status_row)
                print(f"    failed scenario {scenario_name}: {exc}")

    return pd.DataFrame(index_rows), pd.DataFrame(diag_rows), pd.DataFrame(status_rows)


In [ ]:
# ============================================================
# SECTION 6. EXECUTE
# ============================================================

match_dir = resolve_match_dir(match_folder_name)
sample_root = resolve_sample_root(sample_root_dir, match_dir=match_dir)
MUTATION_SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

match_ids = discover_match_ids(match_dir, selected_match_ids=selected_match_ids)
calibration_match_ids = _select_calibration_match_ids(match_ids)

print("Resolved paths:")
print("  Code root               :", CODE_ROOT)
print("  Simulation mutation root:", SIM_MUTATION_ROOT)
print("  Match folder            :", match_dir)
print("  Baseline sample root    :", sample_root)
print("  Mutation samples output :", MUTATION_SAMPLES_DIR)
print()
print(f"Final generation matches  : {len(match_ids)}")
print(f"Calibration matches       : {len(calibration_match_ids)}")
print(f"Enabled families          : {enabled_mutation_families}")
print(f"Preserve marginals mode   : {preserve_marginals_mode}")
print(f"Final physical targeting  : {bool(enforce_final_physical_shift_per_match)}")
if _as_bool(enforce_final_physical_shift_per_match):
    print(f"Targeting tolerance       : {float(final_physical_shift_tolerance):.4f}")
    print(f"Targeting modes to try    : {_physical_target_modes_to_try(preserve_marginals_mode)}")

response_df, selected_df = calibrate_latent_deltas(
    match_ids_for_calibration=calibration_match_ids,
    sample_root=sample_root,
)

scenario_table_df = build_calibrated_scenario_table(selected_df)

sample_index_df, sample_diag_df, sample_status_df = generate_final_samples(
    match_ids=match_ids,
    sample_root=sample_root,
    scenario_table_df=scenario_table_df,
)

# Save metadata and diagnostics.
response_path = MUTATION_SAMPLES_DIR / "Calibration_Response_Curve.csv"
selected_path = MUTATION_SAMPLES_DIR / "Calibration_Selected_Latent_Deltas.csv"
scenario_table_path = MUTATION_SAMPLES_DIR / "Mutation_Sample_Scenario_Table.csv"
sample_index_path = MUTATION_SAMPLES_DIR / "Sample_File_Index.csv"
sample_diag_path = MUTATION_SAMPLES_DIR / "Mutation_Sample_Diagnostics.csv"
sample_status_path = MUTATION_SAMPLES_DIR / "Mutation_Sample_Generation_Status.csv"
config_path = MUTATION_SAMPLES_DIR / "Mutation_Sample_Generation_Config.json"

_write_csv_compact(response_df, response_path, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places)
_write_csv_compact(selected_df, selected_path, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places)
_write_csv_compact(scenario_table_df, scenario_table_path, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places)
_write_csv_compact(sample_index_df, sample_index_path, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places)
_write_csv_compact(sample_diag_df, sample_diag_path, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places)
_write_csv_compact(sample_status_df, sample_status_path, drop_columns=metadata_columns_to_drop, decimal_places=metadata_decimal_places)

config = {
    "code_root": str(CODE_ROOT),
    "simulation_mutation_root": str(SIM_MUTATION_ROOT),
    "match_folder_name": match_folder_name,
    "sample_root_dir": sample_root_dir,
    "mutation_samples_dir": str(MUTATION_SAMPLES_DIR),
    "enabled_mutation_families": list(enabled_mutation_families),
    "desired_physical_shift_abs_levels": [float(x) for x in desired_physical_shift_abs_levels],
    "candidate_latent_abs_grid": [float(x) for x in candidate_latent_abs_grid],
    "selected_match_ids": selected_match_ids,
    "calibration_selected_match_ids": calibration_selected_match_ids,
    "calibration_max_matches": calibration_max_matches,
    "calibration_random_seed": int(calibration_random_seed),
    "calibration_statistic": calibration_statistic,
    "feasibility_tolerance": float(feasibility_tolerance),
    "preserve_marginals_mode": preserve_marginals_mode,
    "enforce_final_physical_shift_per_match": bool(enforce_final_physical_shift_per_match),
    "final_physical_shift_tolerance": float(final_physical_shift_tolerance),
    "physical_target_preserve_modes_to_try": list(physical_target_preserve_modes_to_try) if physical_target_preserve_modes_to_try is not None else None,
    "physical_target_rank_grid_size": int(physical_target_rank_grid_size),
    "physical_target_random_seeds": int(physical_target_random_seeds),
    "physical_target_swap_batches": int(physical_target_swap_batches),
    "physical_target_swaps_per_batch": int(physical_target_swaps_per_batch),
    "physical_target_random_seed": int(physical_target_random_seed),
    "write_mutated_sample_csvs": bool(write_mutated_sample_csvs),
    "include_baseline_in_sample_index": bool(include_baseline_in_sample_index),
    "write_baseline_sample_csvs": bool(write_baseline_sample_csvs),
    "sample_columns_to_save": list(sample_columns_to_save) if sample_columns_to_save is not None else None,
    "nonnegative_sample_columns": list(nonnegative_sample_columns),
    "sample_columns_to_drop": list(sample_columns_to_drop),
    "sample_decimal_places": int(sample_decimal_places),
    "metadata_decimal_places": int(metadata_decimal_places),
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

write_index_workbooks(sample_index_df)

print()
print("Saved calibrated mutation sample outputs:")
print("  Response curve           :", response_path)
print("  Selected latent deltas   :", selected_path)
print("  Scenario table           :", scenario_table_path)
print("  Sample file index        :", sample_index_path)
print("  Sample diagnostics       :", sample_diag_path)
print("  Generation status        :", sample_status_path)
print("  Config                   :", config_path)

print()
print("Selected calibration results:")
display_cols = [
    "mutation_family", "desired_physical_shift", "selected_latent_delta",
    "achieved_calibration_stat_shift", "calibration_error", "calibration_feasible",
    "preserve_marginals_mode", "n_calibration_matches",
]
print(selected_df[display_cols].to_string(index=False))

if _as_bool(enforce_final_physical_shift_per_match):
    print()
    print("Note: selected latent deltas above are dependence-shape starting values. Final per-match physical Pearson targeting is reported below.")

if not sample_status_df.empty:
    print()
    print("Sample-generation status counts:")
    print(sample_status_df["status"].value_counts(dropna=False).to_string())

    if _as_bool(enforce_final_physical_shift_per_match) and "physical_target_error" in sample_status_df.columns:
        mutation_status = sample_status_df.loc[
            sample_status_df["scenario_type"].astype(str).eq("mutation") &
            sample_status_df["status"].astype(str).eq("success")
        ].copy()
        if not mutation_status.empty:
            mutation_status["physical_target_feasible_bool"] = mutation_status["physical_target_feasible"].fillna(False).astype(bool)

            def _mode_summary(values):
                vals = sorted({str(v) for v in values if str(v).strip() != "" and str(v).lower() != "nan"})
                return ",".join(vals)

            target_summary = (
                mutation_status
                .groupby(["mutation_family", "target_physical_shift"], dropna=False)
                .agg(
                    n_matches=("match_id", "count"),
                    n_target_feasible=("physical_target_feasible_bool", "sum"),
                    mean_abs_shift_error=("physical_target_error", "mean"),
                    max_abs_shift_error=("physical_target_error", "max"),
                    min_final_shift=("physical_target_final_shift", "min"),
                    max_final_shift=("physical_target_final_shift", "max"),
                    modes_used=("physical_target_preserve_mode_used", _mode_summary),
                )
                .reset_index()
            )
            print()
            print("Final per-match physical Pearson targeting summary:")
            print(target_summary.to_string(index=False))

            missed = mutation_status.loc[~mutation_status["physical_target_feasible_bool"]].copy()
            if not missed.empty:
                miss_cols = [
                    "match_id", "mutation_family", "target_physical_shift",
                    "physical_target_status", "physical_target_requested_corr",
                    "physical_target_min_attainable_corr", "physical_target_max_attainable_corr",
                    "physical_target_final_shift", "physical_target_error",
                    "physical_target_preserve_mode_used",
                ]
                print()
                print("Rows not within final physical-target tolerance:")
                print(missed[miss_cols].head(50).to_string(index=False))
